# 10. MCH geneplot

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns

from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from ALLCools.mcds import MCDS

from sklearn.metrics import pairwise_distances, roc_auc_score, average_precision_score
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{ENTEX_ROOT}/analysis/mCH_clustering/'


In [ ]:
group_name = 'c16'


In [ ]:
adata = anndata.read_h5ad(f'{outdir}L1/{group_name}/100kCH_embed.h5ad')


In [ ]:
coord_base = 'tsne'
dump_embedding(adata, coord_base)


In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs
cluster_meta = pd.read_csv(f'{indir}clustering/merged/L2final_celltype_L2both_new.tsv', sep='\t', index_col=None, header=0)
clustermap = cluster_meta[['celltype_L2_both_abbr', 'L2_final']].drop_duplicates().set_index('L2_final')['celltype_L2_both_abbr']
meta['celltype_L2_both_abbr'] = meta['L2_final'].astype(str).map(clustermap)
adata.obs['celltype'] = meta['celltype_L2_both_abbr'].astype(str)


In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs
cluster_meta = pd.read_csv(f'{indir}clustering/merged/L2final_celltype_L2both_new.tsv', sep='\t', index_col=None, header=0)
clustermap = cluster_meta[['celltype_L2_both', 'L2_final']].drop_duplicates().set_index('L2_final')['celltype_L2_both']
meta['celltype_L2_both'] = meta['L2_final'].astype(str).map(clustermap)
meta


In [ ]:
meta[['celltype_L2_both', 'Tissue']].to_csv(f'{indir}clustering/merged/5kCG100k3C_subtype_tissue.csv.gz')


In [ ]:
knn = 25
sc.pp.neighbors(adata, n_neighbors=knn, use_rep='100kCH_pca')
sc.tl.leiden(adata, resolution=10, random_state=0, flavor='igraph')


In [ ]:
adata.obs['pseudo'] = adata.obs['leiden'].astype(str) + '-' + adata.obs['celltype'].astype(str)

In [ ]:
obs_pseudo = adata.obs.groupby('pseudo')[['tsne_0', 'tsne_1']].mean()
obs_pseudo['celltype'] = obs_pseudo.index.str.split('-').str[1]
obs_pseudo

In [ ]:
gene_meta_path = f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz'
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'

chrom_to_remove = ['chrL', 'chrM', 'chrX', 'chrY']


In [ ]:
gene_meta = pd.read_csv(gene_meta_path, sep='\t', header=0)
ens2gene = gene_meta.set_index('gene_id')['gene_name'].to_dict()
gene2ens = gene_meta.set_index('gene_name')['gene_id'].to_dict()


In [ ]:
mcds_path_list = np.sort(glob(f'{indir}mcds/*.mcds'))
print(len(mcds_path_list))

In [ ]:
obs_dim = 'cell'
var_dim = 'gene'


In [ ]:
mcds = MCDS.open(mcds_path_list, var_dim=var_dim)
mcds

In [ ]:
mcds = mcds.sel({obs_dim: mcds.get_index('cell').intersection(adata.obs.index)})
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds = mcds.remove_black_list_region(black_list_path=black_list_path, f=0.5)


In [ ]:
mcds


In [ ]:
mcds = mcds.assign_coords(pseudo=('cell', adata.obs.loc[mcds.get_index('cell'), 'pseudo'].values))
mcds = mcds.groupby('pseudo').sum()


In [ ]:
obs_dim = 'pseudo'
mc_type = 'CHN'
mcds = mcds.sel({'mc_type':mc_type})
mcds = MCDS(mcds, obs_dim=obs_dim, var_dim=var_dim)
mcds

In [ ]:
feature_cov_mean = mcds[f'{var_dim}_da'].sel(count_type='cov').mean(dim=obs_dim).to_pandas()
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(feature_cov_mean, bins=100, binrange=(0,1000), ax=ax)


In [ ]:
# use_features = feature_cov_mean[feature_cov_mean > 100].index
# mcds = mcds.sel({var_dim:use_features})
mcds.add_mc_frac(var_dim=var_dim, normalize_per_cell=True, clip_norm_value=10)


In [ ]:
mch_adata = mcds.get_adata(mc_type=mc_type, var_dim=var_dim, select_hvf=False)
# mch_adata = mch_adata[adata.obs.index].copy()
mch_adata = mch_adata[obs_pseudo.index].copy()
mch_adata.obs = obs_pseudo.copy()
mch_adata.var['cov_mean'] = feature_cov_mean.copy()


In [ ]:
def roc_pr(ct):
    roc, pr_pos, pr_neg = [], [], []
    label = mch_adata.obs['celltype'].isin([ct]).astype(int)
    data = mch_adata.X
    for i in range(data.shape[1]):
        roc.append(roc_auc_score(label, data[:, i]))
        pr_pos.append(average_precision_score(label, data[:, i]))
        pr_neg.append(average_precision_score(label, -data[:, i]))
    return [roc, pr_pos, pr_neg]


In [ ]:
# label = []
# for xx in adata.obs['celltype']:
#     if xx.split(' ')[-1].isnumeric():
#         label.append(' '.join(xx.split(' ')[:-1]))
#     else:
#         label.append(xx)
        
# adata.obs['celltype'] = label.copy()
# adata.obs['celltype'].value_counts()

In [ ]:
label = []
for xx in mch_adata.obs['celltype']:
    if xx.split(' ')[-1].isnumeric():
        label.append(' '.join(xx.split(' ')[:-1]))
    else:
        label.append(xx)
        
mch_adata.obs['celltype'] = label.copy()
mch_adata.obs['celltype'].value_counts()


In [ ]:
mch_adata.write_h5ad(f'{outdir}L1/{group_name}/geneCH_pseudo.h5ad')


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

count = mch_adata.obs['celltype'].value_counts()
leg = count.index[count>=10]

cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures, result = {}, {}
    for ct in leg:
        future = executor.submit(roc_pr, ct)
        futures[future] = ct
    
    for future in as_completed(futures):
        ct = futures[future]
        result[ct] = future.result()
        print(f'{ct} finished')


In [ ]:
for ct in leg:
    roc, pr_pos, pr_neg = result[ct]
    print(ct, mch_adata.var.index[np.argsort(pr_pos)[::-1][:5]].map(ens2gene))
    print(ct, mch_adata.var.index[np.argsort(pr_neg)[::-1][:5]].map(ens2gene))


In [ ]:
marker = pd.Index(['LAMP5', 'VIP', 'PVALB', 'SST'])
marker = marker[marker.map(gene2ens).isin(mch_adata.var.index)]
marker_ch = pd.DataFrame(mch_adata[:, marker.map(gene2ens)].X, index=mch_adata.obs.index, columns=marker)


In [ ]:
ncol = 4
nrow = (len(marker)-1) // ncol + 1
ds = 100/np.sqrt(adata.shape[0])
tmp = obs_pseudo.copy()

fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2.4), dpi=300, constrained_layout=True)
for i, xx in enumerate(marker):
    ax = axes.flatten()[i]
    _ = continuous_scatter(ax=ax,
                           data=tmp,
                           hue=marker_ch[xx],
                           scatter_kws={'rasterized':True},
                           coord_base=coord_base,
                           max_points=None,
                           labelsize=8,
                           s=ds,
                           cmap='vlag',
                          )
    ax.set_title(xx)
    
fig.savefig(f'mCH_clustering/{group_name}_pseudo_tsne.gene.pdf', transparent=True)


In [ ]:
ncol = 4
nrow = (len(marker)-1) // ncol + 1
ds = 1000/np.sqrt(adata.shape[0])
tmp = obs_pseudo.copy()

fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2.4), dpi=300, constrained_layout=True)
for i, xx in enumerate(marker):
    ax = axes.flatten()[i]
    _ = continuous_scatter(ax=ax,
                           data=tmp,
                           hue=marker_ch[xx],
                           scatter_kws={'rasterized':True},
                           coord_base=coord_base,
                           max_points=None,
                           labelsize=8,
                           s=ds,
                           cmap='vlag',
                          )
    ax.set_title(xx)
    
fig.savefig(f'mCH_clustering/{group_name}_pseudo_tsne.gene.pdf', transparent=True)


In [ ]:
group_name = 'c13'


In [ ]:
adata = anndata.read_h5ad(f'{outdir}L1/{group_name}/100kCH_embed.h5ad')


In [ ]:
coord_base = 'tsne'
dump_embedding(adata, coord_base)


In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs
cluster_meta = pd.read_csv(f'{indir}clustering/merged/L2final_celltype_L2both_new.tsv', sep='\t', index_col=None, header=0)
clustermap = cluster_meta[['celltype_L2_both_abbr', 'L2_final']].drop_duplicates().set_index('L2_final')['celltype_L2_both_abbr']
meta['celltype_L2_both_abbr'] = meta['L2_final'].astype(str).map(clustermap)
adata.obs['celltype'] = meta['celltype_L2_both_abbr'].astype(str)


In [ ]:
knn = 25
sc.pp.neighbors(adata, n_neighbors=knn, use_rep='100kCH_pca')
sc.tl.leiden(adata, resolution=10, random_state=0, flavor='igraph')


In [ ]:
adata.obs['pseudo'] = adata.obs['leiden'].astype(str) + '-' + adata.obs['celltype'].astype(str)
obs_pseudo = adata.obs.groupby('pseudo')[['tsne_0', 'tsne_1']].mean()
obs_pseudo['celltype'] = obs_pseudo.index.str.split('-').str[1]
obs_pseudo

In [ ]:
# gene_meta_path = f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz'
# black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'

# chrom_to_remove = ['chrL', 'chrM', 'chrX', 'chrY']


In [ ]:
# gene_meta = pd.read_csv(gene_meta_path, sep='\t', header=0)
# ens2gene = gene_meta.set_index('gene_id')['gene_name'].to_dict()
# gene2ens = gene_meta.set_index('gene_name')['gene_id'].to_dict()


In [ ]:
# mcds_path_list = np.sort(glob(f'{indir}mcds/*.mcds'))
# print(len(mcds_path_list))

In [ ]:
obs_dim = 'cell'
var_dim = 'gene'


In [ ]:
mcds = MCDS.open(mcds_path_list, var_dim=var_dim)
mcds

In [ ]:
mcds = mcds.sel({obs_dim: mcds.get_index('cell').intersection(adata.obs.index)})
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds = mcds.remove_black_list_region(black_list_path=black_list_path, f=0.5)


In [ ]:
mcds


In [ ]:
# obs_dim = 'pseudo'
# mcds = mcds.assign_coords(pseudo=('cell', adata.obs.loc[mcds.get_index('cell'), 'pseudo'].values))
# mcds = mcds.groupby(obs_dim).sum()


In [ ]:
mc_type = 'CHN'
mcds = mcds.sel({'mc_type':mc_type})
mcds = MCDS(mcds, obs_dim=obs_dim, var_dim=var_dim)
mcds

In [ ]:
feature_cov_mean = mcds[f'{var_dim}_da'].sel(count_type='cov').mean(dim=obs_dim).to_pandas()
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(feature_cov_mean, bins=100, binrange=(0,1000), ax=ax)


In [ ]:
# use_features = feature_cov_mean[feature_cov_mean > 100].index
# mcds = mcds.sel({var_dim:use_features})
mcds.add_mc_frac(var_dim=var_dim, normalize_per_cell=True, clip_norm_value=10)


In [ ]:
mch_adata = mcds.get_adata(mc_type=mc_type, var_dim=var_dim, select_hvf=False)
mch_adata = mch_adata[adata.obs.index].copy()
mch_adata.obs = adata.obs.copy()
# mch_adata = mch_adata[obs_pseudo.index].copy()
# mch_adata.obs = obs_pseudo.copy()
mch_adata.var['cov_mean'] = feature_cov_mean.copy()


In [ ]:
# def roc_pr(ct):
#     roc, pr_pos, pr_neg = [], [], []
#     label = mch_adata.obs['celltype'].isin([ct]).astype(int)
#     data = mch_adata.X
#     for i in range(data.shape[1]):
#         roc.append(roc_auc_score(label, data[:, i]))
#         pr_pos.append(average_precision_score(label, data[:, i]))
#         pr_neg.append(average_precision_score(label, -data[:, i]))
#     return [roc, pr_pos, pr_neg]


In [ ]:
# label = []
# for xx in adata.obs['celltype']:
#     if xx.split(' ')[-1].isnumeric():
#         label.append(' '.join(xx.split(' ')[:-1]))
#     else:
#         label.append(xx)
        
# adata.obs['celltype'] = label.copy()
# adata.obs['celltype'].value_counts()

In [ ]:
label = []
for xx in mch_adata.obs['celltype']:
    if xx.split(' ')[-1].isnumeric():
        label.append(' '.join(xx.split(' ')[:-1]))
    else:
        label.append(xx)
        
mch_adata.obs['celltype'] = label.copy()
mch_adata.obs['celltype'].value_counts()


In [ ]:
mch_adata.write_h5ad(f'{outdir}L1/{group_name}/geneCH_pseudo.h5ad')


In [ ]:
mch_adata = anndata.read_h5ad(f'{outdir}L1/{group_name}/geneCH_pseudo.h5ad')


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

count = mch_adata.obs['celltype'].value_counts()
leg = count.index[count>=10]

cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures, result = {}, {}
    for ct in leg:
        future = executor.submit(roc_pr, ct)
        futures[future] = ct
    
    for future in as_completed(futures):
        ct = futures[future]
        result[ct] = future.result()
        print(f'{ct} finished')


In [ ]:
for ct in leg:
    roc, pr_pos, pr_neg = result[ct]
    print(ct, mch_adata.var.index[np.argsort(pr_pos)[::-1][:5]].map(ens2gene))
    print(ct, mch_adata.var.index[np.argsort(pr_neg)[::-1][:5]].map(ens2gene))


In [ ]:
marker = pd.Index(['INS', 'GCG', 'SST', 'PPY', 'MAFA', 'PDX1', 'ZC3H3', 'PDE3A'])
marker = marker[marker.map(gene2ens).isin(mch_adata.var.index)]
marker_ch = pd.DataFrame(mch_adata[:, marker.map(gene2ens)].X, index=mch_adata.obs.index, columns=marker)


In [ ]:
ncol = 4
nrow = (len(marker)-1) // ncol + 1
ds = 100/np.sqrt(adata.shape[0])
tmp = adata.obs.copy()

fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2.4), dpi=300, constrained_layout=True)
for i, xx in enumerate(marker):
    ax = axes.flatten()[i]
    _ = continuous_scatter(ax=ax,
                           data=tmp,
                           hue=marker_ch[xx],
                           scatter_kws={'rasterized':True},
                           coord_base=coord_base,
                           max_points=None,
                           labelsize=8,
                           s=ds,
                           cmap='vlag',
                          )
    ax.set_title(xx)
    
fig.savefig(f'mCH_clustering/{group_name}_tsne.gene.pdf', transparent=True)


In [ ]:
ncol = 4
nrow = (len(marker)-1) // ncol + 1
ds = 1000/np.sqrt(adata.shape[0])
tmp = obs_pseudo.copy()

fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2.4), dpi=300, constrained_layout=True)
for i, xx in enumerate(marker):
    ax = axes.flatten()[i]
    _ = continuous_scatter(ax=ax,
                           data=tmp,
                           hue=marker_ch[xx],
                           scatter_kws={'rasterized':True},
                           coord_base=coord_base,
                           max_points=None,
                           labelsize=8,
                           s=ds,
                           cmap='vlag',
                          )
    ax.set_title(xx)
    
fig.savefig(f'mCH_clustering/{group_name}_pseudo_tsne.gene.pdf', transparent=True)
